# Własności obiektów typu `str`

Dzisiaj zajmiemy się przetworzeniem ustrukturyzowanego tekstu prawnego: Konstytucji III RP. 

Będziemy korzystać z własności obiektów typu `str`: https://docs.python.org/3/builtins/stdtypes.html#text-and-binary-sequence-type-methods-summary

Wpierw wczytamy, i spojrzymy na dane zgodnie ze znaną nam z poprzednich zajęć metodologią:

In [ ]:
from pathlib import Path

DATA_DIR = "../../data"
data_file = Path(DATA_DIR) / "others" / "konstytucja.txt"
with open(data_file) as f:
    txt = f.read()

print(f"[...]\n{txt[1000:2000]}\n[...]")

## Tokenizacja

Dla niektórych celów możemy chcieć podzielić tekst na mniejsze jednostki. W momencie gdy jednostki te zgrubsza odpowiadają podziałowi na słowa, możemy nazywać taką procedurę **tokenizacją**, a jej efekty **tokenami**. W ogólniejszym znaczeniu możemy mówić o *segmentacji*.

Naprostszym sposobem podziału na tokeny jest podział po znakach białych (skorzystanie z metody `split` z domyślnymi argumentami):

In [ ]:
tokens = txt.split()
for token in tokens[:50]:
    print(token)

Jakiego typu tokeny możemy zaobserwować w tekście? Spójrzmy wpierw na "kształt" czyli wielkość liter:

In [ ]:
upper_tokens = [tok for tok in tokens if tok.isupper()]
print(upper_tokens)

Innym interesującym nas rodzajem tokenów są te, zapisane z wielkiej litery (ale nie w całości pisane wielkimi literami), nazywamy taki typ `titlecase`:

In [ ]:
title_tokens = [tok for tok in tokens if tok.istitle()]
print(title_tokens[:100])

Metoda `isnumeric` pozwala na wyfiltrowanie tokenów "numerycznych" (co niekoniecznie oznacza sekwencje cyfr):

In [ ]:
num_tokens = [tok for tok in tokens if tok.isnumeric()]
print(num_tokens)

Metoda `isalpha` pozwala (poprzez zaprzeczenie) wypisać tokeny zawierające co najmniej jeden znak niealfabetyczny:

In [ ]:
nonalpha_tokens = [tok for tok in tokens if not tok.isalpha()]
print(nonalpha_tokens[:100])

Przyklejenie znaków interpunkcyjnych do słów jest z punktu widzenia analizy zanieczyszczeniem - to czy po słowie występuje przecinek jest funkcją jego pozycji w zdaniu, a nie wewnętrznej natury.


In [ ]:
PUNCT = """.,()":[]"""

def strip_pun(tok):
    return tok.strip(PUNCT)
    
stripped_tokens = [strip_pun(tok) for tok in tokens]
abnormal_tokens = [tok for tok in stripped_tokens if (not tok.isalpha() and not tok.isnumeric())]
print(abnormal_tokens)

## Wyrażenia regularne

Wyrażenia regularne to język formalny pozwalający na specyfikację abstrakcyjnych wzorców w tekście, które chcemy odnaleźć, i wykorzystać do analizy.

Wyrażenie regularne definiujemy za pomocą stringa. Jeżeli zależy nam na efektywnym przetwarzaniu, warto takie wyrażenie skompilować funkcją `compile`. Najlepiej podawać do kompilacji stringi w postaci surowej, i.e. poprzedzone literką `r`, pozwala to na poprawną interpretację znaków specjalnych (bez konieczności tzw. *escapowania*).

Dokumentacja implementacji wyrażeń regularnych w Pythonie: https://docs.python.org/3/library/re.html

Dobrym zasobem do prób definiowania, i debugowania wyrażeń regularnych jest interaktywny debuger przeglądarkowy, np. https://regex101.com/

In [ ]:
import re

re_polska = re.compile(r"Polska")
re_polska.search(txt)

Podstawowe metody które nas interesują to:
1. `search` znajduje PIERWSZE dopasowanie do wzorca w tekście
1. `match` znajduje dopasowanie o ile występuje ono na samym POCZĄTKU tekstu
1. `findall` znajduje WSZYSTKIE dopasowania do wzorca w tekście

In [ ]:
assert re_polska.match(txt) is None
print(len(re_polska.findall(txt))) # num occurences of "Polska"

Wyrażenia regularne korzystają z odgórnie zdefiniowanych klas, i.e. każdy znak można scharakteryzować jako należący, albo nienależący do takiej klasy, i na tej podstawie definiować wzorce. Interesujące dla nas klasy to w szczególności:

1. `\w` - znaki alfanumeryczne
1. `\d` - cyfry arabskie
2. `\s` - białe znaki
3. `\S` - dopełnienie zbioru białych znaków (negacja `\s`)
4. `\W` - dopełnienie zbioru znaków alfanumerycznych (negacja `\w`)
5. `.` - oznacza dowolny znak (aby uzyskać dopasowanie do kropki, musimy skorzystać z `\.`)

In [ ]:
alpha = re.compile(r"\w")
alpha_b = re.compile("\\w")
assert alpha == alpha_b

print(sorted(set(alpha.findall(txt))))

Interesować nas jednak będą głównie wzorce dłuższe niż jeden znak - w szczególności wzorce o nieokreślonej długości. Najprostszym sposobem definiowania takowych, są *wildcardy*, modyfikują one poprzedzającą frazę wyrażenia, dopuszczając określoną liczbę powtórzeń:
1. `+` oznacza dowolną większą niż 1 liczbę wystąpień
2. `?` oznacza opcjonalne wystąpienie
3. `*` oznacza opcjonalne lub wielokrotne wystąpienie
4. `{n,m}` oznacza min. n-krotne i maks. m-krotne powtórzenie wzorca

Pozwala nam to elegancko wykluczyć "przyklejone" znaki interpunkcyjne, tym samym tworząc doskonalszy tokenizator:

In [ ]:
re_word = re.compile(r"\w+")
re_word.findall(txt)[:20]

Zobaczmy listę wszystkich słów długości 5, w tym celu skorzystamy z dodatkowego znaku specjalnego `\b` oznaczającego krawędź słowa.

In [ ]:
re_word = re.compile(r"\b\w{5}\b")
print(sorted(set(re_word.findall(txt))))

## Ćwiczenie 1:
Zdefiniuj wzorzec pozwalający na odnalezienie wszystkich - dowolnie długich - liczb zapisanych cyframi arabskimi.

Następnie zdefiniuj wzorzec pozwalający odnaleźć wszystkie liczby czterocyfrowe - korespondujące z dużym prawdopodobieństwem do lat.

In [ ]:
#TODO

Aby wykryć liczby rzymskie, musimy skorzystać z klas definiowanych samodzielnie, służą do tego kwadratowe nawiasy:

`[abc]+` - oznacza dowolną, niepustą sekwencję składającą się z liter `abc`, np. "abbaaccab"

Wszystkie liczby rzymskie pasują do pojęcia sekwencji znaków "IVXLCDM", implikacja nie zachodzi w drugą stronę, np. "IIVII" nie jest prawidłową liczbą rzymską.

In [ ]:
re_roman = re.compile(r"\b[IVXLCDM]+\b")

re_roman.findall(txt)

Wykorzystajmy zdobyte umiejętności do segmentacji. Możemy w szczególności chcieć podzielić tekst na rozdziały (segmentację na większe jednostki tekstu, długości ok. jednej strony nazywamy *chunkingiem*):

In [ ]:
re_chapter = re.compile(r"Rozdział\s[IVXLCDM]+\b")

chapters = re_chapter.split(txt)
preamble = chapters[0]
ch1 = chapters[1]

print(preamble)

## Ćwiczenie 2:
Podziel każdy z rodziałów (z osobna!) na artykuły

In [ ]:
#TODO

Artykuły są czasami podzielone na ustępy. Aby wyodrębnić ustępy skorzystamy z dodatkowych funkcjonalności wyrażeń regularnych:
1. `^` oznacza koniec tekstu
2. `$` oznacza koniec tekstu

W standardowym trybie, odnoszą się one do całości stringa, i.e. `^` oznacza początek całego stringa. Możemy jednak wykorzystać flagę, która modyfikuje interpretację wzorca w taki sposób, że każda z linii jest przetwarzana indywidualnie, i wówczas `^` i `$` oznaczają odpowiednio początek i koniec linii.

In [ ]:
re_point_marker = re.compile(r"^\d+\.", re.MULTILINE)
re_point_marker.findall(txt)[:20]

Skorzystamy też z istotniejszej funkcjonalności - grup. Grupy oznaczamy nawiasami okrągłymi, i pozwalają nam one specyfikować "podwzorce", które traktujemy jako całość. 

Na przykład `(abc)+` oznacza powtórzenia grupy `abc`, i.e. dopasuje się do "abcabc", ale już nie "abcab", lub "abbac".

In [ ]:
re_point = re.compile(r"^(\d+)\. (.+)", re.MULTILINE)
match = re_point.search(txt)
print(match)

Metoda `search` zwraca obiekt klasy `Match`, posiadający szereg interesujących funkcjonalności:
1. `start` oznacza początkowy indeks dopasowania
2. `end` oznacza finalny indeks dopasowania
3. `group` jest metodą zwracającą pod-dopasowania do zdefiniowanych (po kolei) grup

In [ ]:
print(match.start(),":", match.end())
print(match.group(1))
print(match.group(2))

## Ćwiczenie 3

Stwórz strukturę danych zawierającą wszystkie ustępy, reprezentowane jako tuple:

(rzymski numer rozdziału, arabski numer artykułu, arabski numer ustępu, treść ustępu).

Skorzystaj z metody `finditer` która pozwala znaleźć wiele dopasowań klasy `Match`, w odróżnieniu od obiektów `str` zwracanych przez `findall`.

In [ ]:
#TODO
#1. odnajdź znaczniki rozdziałów, i na ich podstawie wyznacz granice rozdziałów
#2. w ramach oddziałów odnajdź znaczniki artykułów, i na ich podstawie wyznacz granice artykułów
#3. dla każdego artykułu odnajdź wszystkie występujące w nim ustępy, i oddziel numer ustępu, od jego treści

## Ćwiczenie 4:
Czy jest możliwe napisanie wyrażenia regularnego pozwalającego na wyodrębnienie treści sparowanych nawiasów?

In [ ]:
#TODO